# Phase 3: Graph Diagnostic Attribution Analysis

This notebook loads the `phase3_dataset.csv` generated by the GCP processing pipeline. It trains a LightGBM regression model to map the extracted topological features to the GraphRAG pipeline's F1 score, and uses SHAP to visually explain which structural flaws are causing the most damage.

In [ ]:
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Initialize SHAP JS visualization
shap.initjs()

### 1. Load the Dataset

In [ ]:
# Load the dataset pulled from the GCP VM
df = pd.read_csv('../experiments/runs/phase3_dataset.csv')

print(f"Total queries evaluated: {len(df)}")
display(df.groupby('variant')['f1_score'].mean().to_frame('Mean F1 Score'))

### 2. Train the Attribution Model (LightGBM)

In [ ]:
# Define the features we extracted
features = [
    "node_count", "edge_count", "density", "avg_degree", "component_count", 
    "clustering_coeff", "betweenness_mean", "diameter",
    "seed_confidence_mean", "seed_ambiguity",
    "property_fill_rate", "entity_diversity", "relation_diversity", "property_diversity"
]

# Filter for columns that actually exist in the CSV
X = df[[c for c in features if c in df.columns]]
y = df["f1_score"]

train_data = lgb.Dataset(X, label=y)
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'verbose': -1,
    'seed': 42
}

model = lgb.train(params, train_data, num_boost_round=100)
print("Model trained successfully.")

### 3. Global Feature Importance (SHAP Summary Plot)
This plot shows which features have the biggest impact on the F1 score. 
* **Color:** Shows the feature value (Red = High value, Blue = Low value).
* **X-axis:** Shows the impact on the F1 score (Right = increases F1, Left = decreases F1).

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X, plot_type="dot")

### 4. Local Attribution (Debugging a Specific Failure)
Let's look at a single query that failed on the 'Heavy' degradation variant and see exactly *why* it failed.

In [ ]:
# Find a specific failure in the heavy variant
failed_idx = df[(df['variant'] == 'heavy') & (df['f1_score'] == 0.0)].index[0]

print(f"Query: {df.loc[failed_idx, 'query']}")

# Show the waterfall plot for this specific query
shap.waterfall_plot(shap.Explanation(
    values=shap_values[failed_idx],
    base_values=explainer.expected_value,
    data=X.iloc[failed_idx],
    feature_names=X.columns
))